# ASB Experiment Analysis — Single CSV

Analyse one experiment CSV and compute:
- **ASR** (Attack Success Rate) = `attack_ok` mean
- **TSR** (Task Success Rate) = `task_ok` mean
- **BSR** (Both Success Rate) = `both_ok` mean
- **Wilson 95% CI** for each rate

Results are displayed per agent and as overall aggregates.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# ============================================================
# Configuration — change these before running
# ============================================================
CSV_PATH = "outputs/asb_results_baseline.csv"
EXPERIMENT_LABEL = "Baseline (IPI)"
OUTPUT_STATS = "outputs/single_csv_stats.csv"
# ============================================================

In [ ]:
# ============================================================
# Wilson Score Interval
# ============================================================
def wilson_ci(count: int, n: int, z: float = 1.96):
    """
    Compute the Wilson score interval for a binomial proportion.

    Parameters
    ----------
    count : int
        Number of successes.
    n : int
        Total number of trials.
    z : float
        z-score for the desired confidence level (default 1.96 ≈ 95%).

    Returns
    -------
    tuple[float, float]
        (lower_bound, upper_bound).
    """
    if n == 0:
        return (0.0, 0.0)
    p = count / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return (centre - margin, centre + margin)


def format_rate(val: float) -> str:
    return f"{val:.4f}"


def format_ci(rate: float, lo: float, hi: float) -> str:
    return f"{rate:.4f} [{lo:.4f}, {hi:.4f}]"


print("Functions loaded OK")

In [ ]:
# ============================================================
# Load and filter data
# ============================================================
df = pd.read_csv(CSV_PATH)
total_raw = len(df)

# Keep only successful runs
df = df[df['status'] == 'ok'].copy()
print(f"Loaded: {total_raw} rows ({total_raw - len(df)} filtered as error)")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ============================================================
# Per-agent statistics with Wilson CI
# ============================================================
rows = []
for agent_name, grp in df.groupby('agent_name'):
    n = len(grp)
    a_ok = grp['attack_ok'].sum()
    t_ok = grp['task_ok'].sum()
    b_ok = grp['both_ok'].sum()

    # Rates
    asr = a_ok / n
    tsr = t_ok / n
    bsr = b_ok / n

    # Wilson 95% CI
    asr_ci = wilson_ci(a_ok, n)
    tsr_ci = wilson_ci(t_ok, n)
    bsr_ci = wilson_ci(b_ok, n)

    rows.append({
        'agent_name': agent_name,
        'N': n,
        'ASR': asr,
        'ASR_CI': format_ci(asr, *asr_ci),
        'ASR_lo': asr_ci[0], 'ASR_hi': asr_ci[1],
        'TSR': tsr,
        'TSR_CI': format_ci(tsr, *tsr_ci),
        'TSR_lo': tsr_ci[0], 'TSR_hi': tsr_ci[1],
        'BSR': bsr,
        'BSR_CI': format_ci(bsr, *bsr_ci),
        'BSR_lo': bsr_ci[0], 'BSR_hi': bsr_ci[1],
    })

stats = pd.DataFrame(rows).sort_values('ASR', ascending=False).reset_index(drop=True)
stats.insert(0, 'rank', range(1, len(stats) + 1))
print(f"Computed stats for {len(stats)} agents")

In [ ]:
# ============================================================
# Display: per-agent table
# ============================================================
display(stats[[
    'rank', 'agent_name', 'N',
    'ASR', 'ASR_CI',
    'TSR', 'TSR_CI',
    'BSR', 'BSR_CI',
]].style.format({
    'ASR': '{:.4f}',
    'TSR': '{:.4f}',
    'BSR': '{:.4f}',
}).background_gradient(subset=['ASR', 'TSR', 'BSR'], cmap='RdYlGn_r'))

In [ ]:
# ============================================================
# Overall statistics
# ============================================================
N = len(df)
a_ok = df['attack_ok'].sum()
t_ok = df['task_ok'].sum()
b_ok = df['both_ok'].sum()

asr, tsr, bsr = a_ok / N, t_ok / N, b_ok / N
asr_ci = wilson_ci(a_ok, N)
tsr_ci = wilson_ci(t_ok, N)
bsr_ci = wilson_ci(b_ok, N)

print("=" * 65)
print(f"  Experiment:  {EXPERIMENT_LABEL}")
print(f"  Total cases: {N}  (errors: {(df['status'] == 'error').sum()})")
print("-" * 65)
print(f"  ASR:  {asr:.4f}  [{asr_ci[0]:.4f}, {asr_ci[1]:.4f}]  ({int(a_ok)}/{N})")
print(f"  TSR:  {tsr:.4f}  [{tsr_ci[0]:.4f}, {tsr_ci[1]:.4f}]  ({int(t_ok)}/{N})")
print(f"  BSR:  {bsr:.4f}  [{bsr_ci[0]:.4f}, {bsr_ci[1]:.4f}]  ({int(b_ok)}/{N})")
print("=" * 65)

In [ ]:
# ============================================================
# Visualisation: ASR / TSR / BSR with Wilson error bars
# ============================================================
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

metrics = [
    ('ASR', '#d62728'),
    ('TSR', '#1f77b4'),
    ('BSR', '#2ca02c'),
]

# Sort by ASR for consistent display
plot_df = stats.sort_values('ASR', ascending=True)
y = range(len(plot_df))

fig, axes = plt.subplots(1, 3, figsize=(18, max(6, len(plot_df) * 0.4)))

for ax, (metric, colour) in zip(axes, metrics):
    vals = plot_df[metric].values
    lo = plot_df[f'{metric}_lo'].values
    hi = plot_df[f'{metric}_hi'].values

    err_l = vals - lo
    err_u = hi - vals

    bars = ax.barh(y, vals, height=0.6, xerr=(err_l, err_u),
                   color=colour, edgecolor='white', capsize=3, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['agent_name'], fontsize=8)
    ax.set_xlabel('Rate  [95% Wilson CI]')
    ax.set_title(f'{metric}  [{EXPERIMENT_LABEL}]', fontsize=13, fontweight='bold')
    ax.set_xlim(-0.02, 1.08)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}', va='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Export to CSV
# ============================================================
out = stats[['rank', 'agent_name', 'N',
             'ASR', 'ASR_lo', 'ASR_hi', 'ASR_CI',
             'TSR', 'TSR_lo', 'TSR_hi', 'TSR_CI',
             'BSR', 'BSR_lo', 'BSR_hi', 'BSR_CI']]

out.columns = ['rank', 'agent', 'N',
               'ASR', 'ASR_CI_lo', 'ASR_CI_hi', 'ASR_95%_CI',
               'TSR', 'TSR_CI_lo', 'TSR_CI_hi', 'TSR_95%_CI',
               'BSR', 'BSR_CI_lo', 'BSR_CI_hi', 'BSR_95%_CI']
out.to_csv(OUTPUT_STATS, index=False)
print(f"Exported: {OUTPUT_STATS}")